In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import glob, os, sys
import scanpy as sc
import pandas as pd
import scvi 

print("Last run with scvi-tools version:", scvi.__version__)

Last run with scvi-tools version: 1.3.3


In [3]:
# function for mygene 
import mygene, re, warnings
mg = mygene.MyGeneInfo()

def fetch_chr_mygene(symbols, chunk=1000):
    allowed = {str(i) for i in range(1,20)} | {"X","Y","MT"}
    out = {}
    for i in range(0, len(symbols), chunk):
        chunk_syms = symbols[i:i+chunk]        
        hits = mg.querymany(
            chunk_syms,
            scopes="symbol",
            fields="genomic_pos,genomic_pos_hg19,map_location,symbol",
            species="mouse",
            as_dataframe=False,
            returnall=False,
            verbose=False)
        for h in hits:
            q = h.get("query")
            c = None
            # prefer genomic_pos.chr
            gp = h.get("genomic_pos")
            if isinstance(gp, list) and gp:
                gp = gp[0]
            if isinstance(gp, dict):
                c = gp.get("chr")
            if c is None:
                ml = h.get("map_location")
                if isinstance(ml, str):
                    c = re.split(r"[ ;,]", ml.strip())[0]
            if isinstance(c, str):
                c = c.replace("chr","").upper()
            if c in allowed and q not in out:
                out[q] = c
    return out

# load the output from R scripts

In [6]:
adata_sc_orig = sc.read_h5ad("/root/capsule/data/snRNAseq_LCNE/snRNAseq_LCNE.h5ad")
adata_sc_orig.obs['actualsex'] = adata_sc_orig.obs['sex'].str[0]  # take only first character 'M' or 'F'

# get gene symbols
symbols = adata_sc_orig.var_names.astype(str).tolist()


In [8]:
gene_to_chr = fetch_chr_mygene(symbols)  # get the chromosomal info!
adata_sc_orig.var["chromosome"] = adata_sc_orig.var_names.map(gene_to_chr).astype("category")


# lets just save this so that we dont need to run query all the time 
adata_sc_orig.write("/root/capsule/scratch/snRNAseq_LCNE_with_chrom.h5ad")
